# 📊 Pocket OTC AI Analyzer — Google Colab

يشغّل واجهة الويب وTelegram Bot معًا.

**READ-ONLY:** لا يسجل الدخول إلى Pocket Option ولا ينفذ أوامر تداول.

⚙️ خلية التشغيل تقوم تلقائيًا بإعادة تنزيل المشروع إذا كان مجلد `/content/Jjjjjjj` مفقودًا.

In [ ]:
# تجهيز المشروع تلقائيًا — يمكن تشغيل هذه الخلية أكثر من مرة بأمان
import os, shutil, subprocess, sys

ROOT = '/content/Jjjjjjj'
REPO = 'https://github.com/mohmb142/Jjjjjjj.git'

def ensure_project():
    if not os.path.isfile(os.path.join(ROOT, 'main.py')):
        print('📥 المشروع غير موجود أو غير مكتمل — سيتم تنزيل نسخة GitHub الأخيرة...')
        if os.path.isdir(ROOT):
            shutil.rmtree(ROOT, ignore_errors=True)
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
    else:
        print('✅ المشروع موجود، سيتم استخدام النسخة الحالية في الجلسة.')
    if not os.path.isfile(os.path.join(ROOT, 'main.py')):
        raise FileNotFoundError(f'لم يتم العثور على main.py داخل {ROOT}')
    os.chdir(ROOT)
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)

ensure_project()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'colab_requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', '.'], check=True)
print(f'✅ المشروع جاهز: {ROOT}')
print('✅ تم تثبيت المتطلبات وفحص ملفات Python')

In [ ]:
import os
from getpass import getpass

telegram_token = getpass('🔐 Telegram Bot Token: ').strip()
openrouter_key = getpass('🔐 OpenRouter API Key: ').strip()

if not telegram_token:
    raise ValueError('Telegram Bot Token مطلوب')
if not openrouter_key:
    raise ValueError('OpenRouter API Key مطلوب')

os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token
os.environ['OPENROUTER_API_KEY'] = openrouter_key
os.environ['OPENROUTER_MODEL'] = 'google/gemini-2.5-flash'
print('✅ تم إعداد المفاتيح داخل جلسة Colab فقط')

In [ ]:
# تشغيل Web UI + Telegram Bot مع إصلاح تلقائي لمسار المشروع
import os, socket, threading, time, traceback, sys, subprocess, shutil, uvicorn
from google.colab.output import eval_js
from IPython.display import HTML, display

ROOT = '/content/Jjjjjjj'
REPO = 'https://github.com/mohmb142/Jjjjjjj.git'
PORT = 8000

# حماية إضافية: إذا حُذف المجلد لأي سبب، أعد تنزيله تلقائيًا قبل import main.
if not os.path.isfile(os.path.join(ROOT, 'main.py')):
    print('⚠️ مجلد المشروع مفقود — إعادة clone تلقائيًا...')
    if os.path.isdir(ROOT):
        shutil.rmtree(ROOT, ignore_errors=True)
    subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

try:
    import main
    print('✅ main.py تم استيراده بنجاح')
    print('✅ FastAPI app:', main.app)
except Exception as exc:
    print('❌ خطأ أثناء استيراد main.py:')
    traceback.print_exc()
    raise RuntimeError(f'فشل استيراد main.py: {exc}') from exc

def port_ready(host='127.0.0.1', port=PORT, timeout=30):
    end = time.time() + timeout
    while time.time() < end:
        try:
            with socket.create_connection((host, port), timeout=0.5):
                return True
        except OSError:
            time.sleep(0.25)
    return False

def web_worker():
    try:
        uvicorn.run(main.app, host='0.0.0.0', port=PORT, log_level='info')
    except Exception:
        print('❌ Web UI error:')
        traceback.print_exc()

if 'web_thread' not in globals() or not web_thread.is_alive():
    web_thread = threading.Thread(target=web_worker, daemon=True, name='fastapi-web')
    web_thread.start()

if not port_ready():
    print('❌ FastAPI لم تبدأ على المنفذ 8000.')
    raise RuntimeError('❌ لم تبدأ FastAPI على المنفذ 8000 — راجع traceback أعلاه')

public_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
html = f'''
<div style="padding:24px;border:1px solid #ddd;border-radius:12px;font-family:Arial,sans-serif;margin:10px 0">
<h2>📊 Pocket OTC AI Analyzer</h2>
<p>✅ واجهة تحليل صور الشارت تعمل الآن.</p>
<a href="{public_url}" target="_blank" style="display:inline-block;padding:14px 24px;background:#1976d2;color:#fff;text-decoration:none;border-radius:8px;font-weight:bold">🚀 فتح الواجهة</a>
<p style="margin-top:15px;word-break:break-all"><b>الرابط العام:</b><br>{public_url}</p>
</div>
'''
display(HTML(html))
print('🌐 الرابط العام:', public_url)

from telegram_bot import run

def telegram_worker():
    try:
        print('🤖 بدء Telegram Bot...')
        run()
    except Exception:
        print('❌ Telegram Bot error:')
        traceback.print_exc()

if 'telegram_thread' not in globals() or not telegram_thread.is_alive():
    telegram_thread = threading.Thread(target=telegram_worker, daemon=True, name='telegram-bot')
    telegram_thread.start()

print('🤖 Telegram Bot: يعمل في الخلفية')
print('🌐 Web UI: تعمل في الخلفية')
print('✅ Web UI + Telegram يعملان معًا')

In [ ]:
# اختبار الواجهة
import requests
response = requests.get('http://127.0.0.1:8000/health', timeout=10)
print('HTTP status:', response.status_code)
print('Response:', response.json())
if response.status_code != 200:
    raise RuntimeError('❌ فشل اختبار FastAPI')
print('✅ FastAPI health check نجح')
print('🌐 الرابط العام:', public_url)
print('⚠️ لا تغلق جلسة Colab حتى يبقى الرابط والبوت يعملان.')